# Candidate Generation: Content Baselines

Personalised content baselines that rank each request by **cosine similarity** to its causal history instead of split-wide popularity. Three variants are built: **text**, **entity** and a **text/entity fusion**. They reuse the causal per-request profiles saved by `4_Feature_Engineering` (`full_history_rows`) and write the standard candidate table so they can be compared with the popularity controls.

Protocol:

- Load the causal request profiles saved by stage 4 (`full_history_rows`); they are the request-time state before current feedback.
- Score time-eligible articles by cosine similarity using the frozen stage 3 text and entity vectors.
- Fuse text and entity scores with equal weights, renormalising when only one side is valid.
- Restrict candidates to `available_from <= request timestamp` and exclude every article in the request's `full_history_rows`.
- Break ties by training exposure popularity descending, then ascending `news_code`.
- Fall back to the training click-popularity order (clicks descending, exposures tie-break) for empty or zero profiles.
- Write the standard `impression_uid, cutoff, news_code` table and evaluate it with `cg_evaluation.py`.


## 1. Setup

In [1]:
import os
import sys
from pathlib import Path

import numpy as np
import polars as pl


CODE_ROOT = next(
    p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'run_all.py').is_file()
)
ROOT = Path(os.environ.get('NEWSREC_ROOT', CODE_ROOT)).expanduser().resolve()
sys.path.insert(0, str(CODE_ROOT / '5_Models' / '5_01_Candidate_Generation'))

from cg_evaluation import CandidateEvaluator
from cg_result_writer import CandidateWriter

PROCESSED = ROOT / '1_Data_Preparation' / 'processed'
FEATURES = ROOT / '4_Feature_Engineering' / 'artifacts' / 'prequential-all'
REPRESENTATIONS = ROOT / '3_Representations' / 'artifacts'
ARTIFACTS = ROOT / '5_Models' / '5_01_Candidate_Generation' / 'artifacts'
RESULTS = ROOT / 'results'

TARGET_SPLIT = 'validation'
M_VALUES = (50, 100, 200, 500)
CANDIDATE_CAP = max(M_VALUES)
MODELS = ('content_text', 'content_entity', 'content_text_entity')

print('Repository:', ROOT)
print('Target split:', TARGET_SPLIT)
print('M values:', M_VALUES)
print('Candidate cap:', CANDIDATE_CAP)

Repository: C:\Users\Yong Rui\BT4222-Group-6-
Target split: validation
M values: (50, 100, 200, 500)
Candidate cap: 500


## 2. Catalogue and causal request profiles

Article rows match the shared `article_index` convention (`article_row = 1..N`); the padding row `0` is never eligible. The request profiles are the full causal history used by stage 4, i.e. the request-time state before current feedback.


In [2]:
profile_parts = sorted((FEATURES / f'request_profiles_{TARGET_SPLIT}').glob('*.parquet'))
profiles = pl.read_parquet(
    profile_parts,
    columns=['impression_uid', 'timestamp', 'full_history_rows'],
)

index = pl.read_parquet(
    REPRESENTATIONS / 'article_index.parquet',
    columns=['article_row', 'news_code'],
)
n_articles = index.height + 1

profiles = profiles.sort('timestamp')
history_rows = profiles['full_history_rows'].to_list()
timestamps = profiles['timestamp'].to_numpy()
impression_uids = profiles['impression_uid'].to_numpy()

print(f'Requests: {profiles.height:,}')
print(f'Unique timestamps: {profiles["timestamp"].n_unique():,}')
print(f'Catalogue articles: {n_articles - 1:,}')
print(f'Requests with empty causal history: {sum(not rows for rows in history_rows):,}')
print(f'Mean history length: {np.mean([len(rows) for rows in history_rows]):.2f}')

Requests: 31,624
Unique timestamps: 24,580
Catalogue articles: 51,282
Requests with empty causal history: 114
Mean history length: 38.95


## 3. Article representations, availability and the click-popularity fallback

Text and entity vectors are the frozen stage 3 representations. Each article is joined to its observable `available_from` so articles can be ordered into an eligible prefix per timestamp. The fallback order is the training click-popularity order (clicks descending, exposures as tie-break), which is used only for cold starts.


In [3]:
text = np.load(REPRESENTATIONS / 'text64.npy', mmap_mode='r')
entity = np.load(REPRESENTATIONS / 'article_entity100.npy', mmap_mode='r')

availability = pl.read_parquet(
    REPRESENTATIONS / 'article_availability.parquet',
    columns=['news_code', 'available_from'],
)
catalogue = index.join(availability, on='news_code', how='left')

counts = pl.read_parquet(
    PROCESSED / 'news_catalogue.parquet',
    columns=['news_code', 'exposures_train', 'clicks_train'],
)
popularity = counts.select(
    'news_code',
    pl.col('exposures_train').alias('popularity_score'),
    pl.col('clicks_train').alias('click_score'),
)
catalogue = catalogue.join(popularity, on='news_code', how='left')

ordered = catalogue.sort(['available_from', 'news_code'])
article_rows = ordered['article_row'].to_numpy().astype(np.int64)
news_sorted = ordered['news_code'].to_numpy().astype(np.int64)
available_sorted = ordered['available_from'].to_numpy().astype('datetime64[ns]')
popularity_sorted = ordered['popularity_score'].to_numpy().astype(np.int64)
click_sorted = ordered['click_score'].to_numpy().astype(np.int64)
text_sorted = np.asarray(text[article_rows], dtype=np.float32)
entity_sorted = np.asarray(entity[article_rows], dtype=np.float32)
text_article_valid = np.linalg.norm(text_sorted, axis=1) > 0
entity_article_valid = np.linalg.norm(entity_sorted, axis=1) > 0

pos_in_sorted = np.empty(n_articles, dtype=np.int64)
pos_in_sorted[article_rows] = np.arange(len(article_rows), dtype=np.int64)
global_click_order = np.lexsort(
    (news_sorted, -popularity_sorted, -click_sorted)
)

print(f'Text article vectors: {int(text_article_valid.sum()):,} / {len(article_rows):,}')
print(f'Entity article vectors: {int(entity_article_valid.sum()):,} / {len(article_rows):,}')
print(f'Click fallback scores: {len(click_sorted):,}')

Text article vectors: 51,282 / 51,282
Entity article vectors: 44,265 / 51,282
Popularity scores: 51,282


## 4. Personalised content scorer

`pool_profiles` mean-pools the valid history vectors per request and L2-normalises them, so empty or all-zero histories produce a zero profile. Requests are then scored by cosine similarity against the eligible catalogue; the fused score is `0.5 * text_cosine + 0.5 * entity_cosine` when both sides are valid and the valid side alone otherwise. `select_top_k` applies the history exclusion and the deterministic tie-breaks.


In [4]:
def pool_profiles(history_rows, matrix, block=10000):
    '''Mean-pool valid article vectors per request and L2-normalise.'''
    matrix = np.asarray(matrix)
    n = len(history_rows)
    dim = matrix.shape[1]
    result = np.zeros((n, dim), dtype=np.float32)

    for start in range(0, n, block):
        chunk = history_rows[start : start + block]
        lengths = np.fromiter(
            (len(rows) for rows in chunk), dtype=np.int64, count=len(chunk)
        )
        flat = np.concatenate(
            [np.asarray(rows, dtype=np.int64) for rows in chunk if len(rows)]
        )
        vectors = np.asarray(matrix[flat], dtype=np.float32)
        valid = np.linalg.norm(vectors, axis=1) > 1e-12

        groups = np.repeat(np.arange(len(chunk), dtype=np.int64), lengths)
        group_valid = groups[valid]
        vector_valid = vectors[valid]
        counts = np.bincount(group_valid, minlength=len(chunk))
        sums = np.empty((len(chunk), dim), dtype=np.float32)
        for column in range(dim):
            sums[:, column] = np.bincount(
                group_valid,
                weights=vector_valid[:, column],
                minlength=len(chunk),
            )
        present = counts > 0
        pooled = sums[present] / counts[present, None]
        norms = np.linalg.norm(pooled, axis=1)
        pooled /= np.where(norms > 0, norms, 1.0)[:, None]
        result[start : start + len(chunk)][present] = pooled

    return result

In [5]:
text_profiles = pool_profiles(history_rows, text)
entity_profiles = pool_profiles(history_rows, entity)
text_profile_valid = np.linalg.norm(text_profiles, axis=1) > 0
entity_profile_valid = np.linalg.norm(entity_profiles, axis=1) > 0

print(f'Valid text profiles: {int(text_profile_valid.sum()):,} / {len(text_profiles):,}')
print(f'Valid entity profiles: {int(entity_profile_valid.sum()):,} / {len(entity_profiles):,}')
print(f'No text profile: {int((~text_profile_valid).sum()):,}')
print(f'No entity profile: {int((~entity_profile_valid).sum()):,}')

Valid text profiles: 31,510 / 31,624
Valid entity profiles: 31,454 / 31,624
No text profile: 114
No entity profile: 170


In [6]:
def fuse_scores(text_scores, entity_scores, text_available, entity_available):
    '''Equal-weight fusion with valid-side renormalisation.'''
    fused = np.full(text_scores.shape, -np.inf, dtype=np.float32)
    both = text_available & entity_available
    only_text = text_available & ~entity_available
    only_entity = entity_available & ~text_available
    fused[both] = 0.5 * (text_scores[both] + entity_scores[both])
    fused[only_text] = text_scores[only_text]
    fused[only_entity] = entity_scores[only_entity]
    return fused

In [7]:
def popularity_select(order, n_eligible, excluded, k):
    '''Take popular eligible positions from a precomputed global order.'''
    excluded_set = set(map(int, excluded))
    selected = []
    for position in order:
        position = int(position)
        if position < n_eligible and position not in excluded_set:
            selected.append(position)
            if len(selected) == k:
                break
    return np.asarray(selected, dtype=np.int64)

In [8]:
def select_top_k(scores, popularity, news_code, k, excluded, fallback_order=None):
    '''Deterministic Top-k by score, popularity, then news_code.'''
    n = scores.size
    if fallback_order is not None:
        return popularity_select(fallback_order, n, excluded, k)

    positions = np.asarray(excluded, dtype=np.int64)
    if positions.size:
        scores = scores.copy()
        scores[positions] = -np.inf

    frontier = np.argpartition(-scores, k - 1)[:k]
    threshold = scores[frontier].min()
    above = np.flatnonzero(scores > threshold)
    need = k - above.size
    ties = np.flatnonzero(scores == threshold)
    if ties.size > need:
        tie_order = np.lexsort((news_code[ties], -popularity[ties]))
        ties = ties[tie_order[:need]]
    chosen_positions = np.r_[above, ties]

    order = np.lexsort(
        (
            news_code[chosen_positions],
            -popularity[chosen_positions],
            -scores[chosen_positions],
        )
    )
    return chosen_positions[order].astype(np.int64)

## 5. Generate the standard candidate table

Requests are grouped by timestamp. Each request scores the whole eligible catalogue, excludes its full causal history, and writes the Top-500 `impression_uid, cutoff, news_code` rows in the shared long format. Empty or zero profiles fall back to the click-popularity order.


In [9]:
cutoff_values = np.arange(1, CANDIDATE_CAP + 1, dtype=np.int16)
paths = {
    model: ARTIFACTS
    / model
    / f'recommendations_{TARGET_SPLIT}_top{CANDIDATE_CAP}.parquet'
    for model in MODELS
}
writers = {model: CandidateWriter(path) for model, path in paths.items()}
starts = np.flatnonzero(np.r_[True, timestamps[1:] != timestamps[:-1]])
ends = np.r_[starts[1:], len(timestamps)]
batches = list(zip(starts, ends))

print(f'Scoring requests: {len(timestamps):,}')

for batch_no, (start, end) in enumerate(batches, 1):
    timestamp = timestamps[start]
    n_eligible = int(np.searchsorted(available_sorted, timestamp, side='right'))

    popularity_batch = popularity_sorted[:n_eligible]
    news_batch = news_sorted[:n_eligible]
    text_batch = text_sorted[:n_eligible]
    entity_batch = entity_sorted[:n_eligible]
    text_article_available = text_article_valid[:n_eligible]
    entity_article_available = entity_article_valid[:n_eligible]

    text_scores = text_profiles[start:end] @ text_batch.T
    entity_scores = entity_profiles[start:end] @ entity_batch.T

    text_available = (
        text_profile_valid[start:end, None] & text_article_available[None, :]
    )
    entity_available = (
        entity_profile_valid[start:end, None] & entity_article_available[None, :]
    )
    text_scores[~text_available] = -np.inf
    entity_scores[~entity_available] = -np.inf
    fused_scores = fuse_scores(
        text_scores, entity_scores, text_available, entity_available
    )

    batch_size = end - start
    selected_batch = {
        model: np.empty((batch_size, CANDIDATE_CAP), dtype=np.int64)
        for model in MODELS
    }
    for local, profile_row in enumerate(range(start, end)):
        rows = history_rows[profile_row]
        excluded = (
            pos_in_sorted[np.asarray(rows, dtype=np.int64)]
            if rows
            else np.empty(0, dtype=np.int64)
        )
        excluded = excluded[excluded < n_eligible]

        for model, scores, fallback in (
            (
                'content_text',
                text_scores[local],
                None if text_profile_valid[profile_row] else global_click_order,
            ),
            (
                'content_entity',
                entity_scores[local],
                None if entity_profile_valid[profile_row] else global_click_order,
            ),
            (
                'content_text_entity',
                fused_scores[local],
                None
                if (text_profile_valid[profile_row] or entity_profile_valid[profile_row])
                else global_click_order,
            ),
        ):
            selected_batch[model][local] = select_top_k(
                scores,
                popularity_batch,
                news_batch,
                CANDIDATE_CAP,
                excluded,
                fallback,
            )

    uids = impression_uids[start:end]
    expanded_uid = np.repeat(uids, CANDIDATE_CAP)
    expanded_cutoff = np.tile(cutoff_values, batch_size)
    for model in MODELS:
        writers[model].add(
            expanded_uid,
            expanded_cutoff,
            news_batch[selected_batch[model].reshape(-1)],
        )

    if batch_no % 5000 == 0 or batch_no == len(batches):
        print(f'Scored {batch_no:,} / {len(batches):,} timestamp batches', flush=True)

for writer in writers.values():
    writer.close()

for model, path in paths.items():
    print(f'{model}: {writers[model].rows:,} candidate rows -> {path}')

Scoring requests: 31,624
Scored 5,000 / 24,580 timestamp batches
Scored 10,000 / 24,580 timestamp batches
Scored 15,000 / 24,580 timestamp batches
Scored 20,000 / 24,580 timestamp batches
Scored 24,580 / 24,580 timestamp batches
content_text: 15,812,000 candidate rows -> C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\artifacts\content_text\recommendations_validation_top500.parquet
content_entity: 15,812,000 candidate rows -> C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\artifacts\content_entity\recommendations_validation_top500.parquet
content_text_entity: 15,812,000 candidate rows -> C:\Users\Yong Rui\BT4222-Group-6-\5_Models\5_01_Candidate_Generation\artifacts\content_text_entity\recommendations_validation_top500.parquet


## 6. Evaluation

Uses the shared `CandidateEvaluator`, which checks unique candidates, complete cutoffs and full impression coverage. It reports recall (pooled and impression-mean), hit rate, long-tail recall, cold-article recall, novelty (bits), catalogue coverage and personalisation for each model and M, and writes `results/candidate_generation_<model>_<split>.csv`.


In [11]:
evaluator = CandidateEvaluator(root=ROOT, split=TARGET_SPLIT)
evaluation_paths = {
    model: RESULTS / f'candidate_generation_{model}_{TARGET_SPLIT}.csv'
    for model in MODELS
}
for model in MODELS:
    evaluator.evaluate_to_csv(
        paths[model],
        model_name=model,
        m_values=M_VALUES,
        output_dir=RESULTS,
    )

pl.concat([pl.read_csv(path) for path in evaluation_paths.values()]).sort(['m', 'model'])

model,split,m,metric,value,numerator,denominator
str,str,i64,str,f64,f64,f64
"""content_entity""","""validation""",50,"""recall_pooled""",0.00392,188.0,47961.0
"""content_entity""","""validation""",50,"""recall_impression_mean""",0.004265,null,31624.0
"""content_entity""","""validation""",50,"""personalisation""",0.971042,9710.421522,10000.0
"""content_entity""","""validation""",50,"""novelty_bits""",15.710183,2.4841e7,1.5812e6
"""content_entity""","""validation""",50,"""long_tail_recall""",0.006238,61.0,9778.0
…,…,…,…,…,…,…
"""content_text_entity""","""validation""",500,"""long_tail_recall""",0.039681,388.0,9778.0
"""content_text_entity""","""validation""",500,"""novelty_bits""",15.708171,2.4838e8,1.5812e7
"""content_text_entity""","""validation""",500,"""personalisation""",0.925434,9254.337243,10000.0


## Notes and limitations

- The profiles are causal and pre-feedback for each request, but they consume earlier validation clicks. The rolling popularity baseline is also prequential, so this is a content-versus-popularity comparison using earlier feedback on both sides.
- Excluding the full causal history deliberately removes repeat-click opportunities from the candidate set. The logged slates contain a small amount of history overlap, so recall on repeat clicks is not measured here.
- Text vectors are fitted on the training catalogue; entity vectors are frozen pretrained vectors. Both are shared across splits.
- Users with no valid profile fall back to click popularity, which keeps cold starts safe.
- `available_from` is the project-wide observable-availability proxy, not a verified publication timestamp.
